# Guide to use my functions calculating X-ray luminosity from binaires

> This notebook is here to guide the user that would like to work with X-ray binary systems. We have defined a new set of functions used to sort binary systems, to calculate the X-ray luminosity of the filtered systems and to visualize the cumulative X-ray luminosity function of a given population. Systems of interest are those including a compact object with a main-sequence or post-main sequence companion star.

<hr style="border: none; height: 3px;
background: linear-gradient(to right, #0f23fa, #0fe7fa);">

## Contents

**Workflow summary**

1. Import and setup  
2. Systems selection
    - Single population
    - Multiple populations
3. X-ray luminosity calculations
    - Loading the population
    - The Transient population DataFrame
4. Exploring the synthetic XRB population
    - Weight calculation
    - Visualizing the synthetic population
    - Visualizing a sub-population from a global population

</div>

<hr style="border: none; height: 3px;
background: linear-gradient(to right, #0f23fa, #0fe7fa);">

## 1) Import and setup

To run the functions properly without error, we need to import some required modules.

<div align="left">

```python
import os

from scipy.optimize import newton
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from posydon.popsyn.synthetic_population import Population
from posydon.utils.common_functions import orbital_separation_from_period, roche_lobe_radius
from posydon.utils.posydonwarning import Pwarn, POSYDONWarning
import posydon.utils.constants as const
```

</div>

<hr style="border: none; height: 3px;
background: linear-gradient(to right, #0f23fa, #0fe7fa);">

## 2) Systems selection 



 After simulations are done, a `.h5` file is created. From this file, we would extract the system containing one compact object (either a neutron star or a black hole) with a H-rich or He-rich companion in order to further calculate their X-ray luminosity. For that you can use the function `Systems_selection_for_XRB()` that will filter your population. Although the use of the function is describe in the DocString, here is an example on how to use it:

### 2.1 Single population

If only a single file need to be filtered, do as follow: 

<br>

<div align="center">

```python
Systems_selection_for_XRB('path_to_population/filename.h5', 'New_filename', verbose=False)
```

</div>  
<br>

<div style="
padding: 12px;
border-left: 4px solid #0ea5e9;
background-color: rgba(14,165,233,0.08);
border-radius: 6px;
">

<b>Note:</b>  
The output filename must not contain the extension `.h5` at the end. It will be automatically added. In general, on computational cluster, it takes ~ 8 minutes for 10'000'000 binaries to be filtered. The newly created file is stored in the same folder as the input `.h5` file.

</div>

The `verbose` parameter only print information about the current process. 

<details>
<summary> Result if True </summary>

```text
Processing file `filename.h5` in Popualtion()...

Exporting selected systems to `path_to_population/New_filename.h5`
```
</details>

At the end, you will get a resume of the process including the number of selected system as well as the process execution time. Below is an example of output.

<details>
<summary> Show output </summary>

```text
======================================================= 
  📁 Input  : filename.h5 
  💾 Output : Name_of_the_newly_created_file.h5 
─────────────────────────────────────────────────────── 
  Total systems (input):         10000000 
  Selected systems (output):     17500 
  Removed systems:               9982500 
─────────────────────────────────────────────────────── 
  Execution time:                8min 36sec 
======================================================= 
```

</details>

### 2.2 Multiple populations

It is also possible to filter multiple files at once as follow:

<br>
<div align="center">

```python
files_list = ['path_to_file_1/population1.h5', 'path_to_file_2/population2.h5', 'path_to_file_3/population3.h5']
filenames_out_list = ['Newname1', 'Newname2', 'Newname3']

Systems_selection_for_XRB(files_list, filenames_out_list, verbose=False)

```

</div>
<br>

<details>
<summary> Show output </summary>

```text
======================================================= 
  📁 Input  : population1.h5 
  💾 Output : Newname1.h5 
─────────────────────────────────────────────────────── 
  Total systems (input):         10000000 
  Selected systems (output):     15500 
  Removed systems:               9985500 
─────────────────────────────────────────────────────── 
  Execution time:                8min 22sec 
======================================================= 


======================================================= 
  📁 Input  : population1.h5 
  💾 Output : Newname1.h5 
─────────────────────────────────────────────────────── 
  Total systems (input):         2500000
  Selected systems (output):     7650 
  Removed systems:               2492350 
─────────────────────────────────────────────────────── 
  Execution time:                5min 14sec 
======================================================= 


======================================================= 
  📁 Input  : population3.h5 
  💾 Output : Newname3.h5 
─────────────────────────────────────────────────────── 
  Total systems (input):         100000 
  Selected systems (output):     3756 
  Removed systems:               96244 
─────────────────────────────────────────────────────── 
  Execution time:                1min 04sec 
======================================================= 
```
</details>


<hr style="border: none; height: 3px;
background: linear-gradient(to right, #0f23fa, #0fe7fa);">

## 3) X-ray Luminosity calculations


Using the newly created file containing systems with a compact object (Black hole or neutron star) accompanied by a star rich in hydrogen or helium, we proceed to calculate the luminosity.

### 3.1 Loading the population

First, we import the function `Population()` and then we load our new selected population in it. 

<div align="center">

```python
Pop = Population('New_Filename.h5')
```

</div>

Then we need to calculate the formation channels of the above population. To achieve this, we use 
<div align="center">

```python  
Pop.calculate_formation_channels(mt_history=True)
```

</div>

<div style="
padding: 12px;
border-left: 4px solid #0ea5e9;
background-color: rgba(14,165,233,0.08);
border-radius: 6px;
">

<b>Tip:</b>  
Compute a population's formation channels only once ! Once calculated, the channels are stored in the `.h5` file. If the population in reloaded through `Population()` — even in a different project or a later session — the formation channels are still available, without needing to be recomputed.
</div>


### 3.2 The transient population DataFrame

Finally, we pass the population into the function that creates the transient popualtion. If no keyword argument is provided to the selection function, then we create the transient population as follow:

<div align="center">

```python

Transient_Pop = Pop.create_transient_population(XRB_selection_function, 'Pop_trans_name')

```

</div>

<div style="
padding: 12px;
border-left: 4px solid #0ea5e9;
background-color: rgba(14,165,233,0.08);
border-radius: 6px;
">

<b>Error note:</b> 

The DocString of the `XRB_selection_function()` function provides a list of the columns required to calculate luminosity; if any of these columns are missing, an error occurs.

</div>



If keyword arguments (**kwargs) need to be passed to the selection function (see DocString of `XRB_selection_function()` for supported kwargs), their input must be:

<div align="center">

```python
Transient_Pop = Pop.create_transient_population(lambda h, o, f: XRB_selection_function(h, o, f, **kwargs), 'Pop_trans_name')
```

</div>

The `XRB_selection_function()` does everything needed to get the X-ray luminosity of binaries and outputs a DataFrame. The Dataframe's columns contains the state at the end of the simualtion of each system and below are described the columns of the DataFrame.

<details>
<summary> DataFrame columns </summary>

- `metallicity`: metallicity that is used to create the HMS-HMS stars of the system
- `time`: the age of the system when the simulation ends
- `Binary_state`: The state of the system (RLO1/2 - detached - Be system)
- `Accretor_state`: Either black hole or neutron star
- `Accretor_spin`: the spin of the accretor $\in [0, 1]$ (dimensionless)
- `Accretor_mass`: The mass of the accretor in [$M_{\odot}$]
- `Accretor_logR`: Accretor's radius in log10(Radius) [log10($R_{\odot}$)]
- `Donor_state`: Main sequence star (H-rich), post-MS star (He-rich)
- `Donor_mass`: Mass of the donor star [$M_{\odot}$]
- `Donor_LogR`: Donor's radius in log10(Radius) [log10($R_{\odot}$)]
- `Donor_Roche_lobe_radius`: Roche lobe radius of the donor star in $R_{\odot}$.
- `Donor_LogL`: Surface luminosity of the donor star in [$L_{\odot}$]
- `Donor_H1_at_surface`: Hydrogen abundance at the surface of the donor star. Specifically used for the calculation of mass accretion rate of wind-fed systems.
- `Donor_He_core_mass`: Mass of the helium core of the donor star in [$M_{\odot}$]
- `Accretor_LogMdot`: Final absolute mass cahnge of the accretor due to RLO of wind in [log10($M_{\odot}/yr$)]
- `Donor_LogMdot_wind`: Mass lost by wind from the donor star in [log10($M_{\odot}/yr$)]
- `lg_mtransfer_rate`: Mass transfer rate through $L_1$ due to Roche lobe overflow in [$M_{\odot}/yr$].
- `Orbital_separation`: Orbital separation between the two objects in [$R_{\odot}$]
- `Eccentricity`: Eccentricity of the system
- `System_type`:  Type of system (RLO - Wind - Be)
- `Eddington_state`: The Eddington state of the binary (Sub-Eddington or Super-Eddington).
- `Beaming`: Geometrical beaming of the system due to the presence of an accretion disk. 
- `Weight_x_beaming`: Boolean value. If False, means that the weight already includes the beaming factor and therefore we do not need to multiply the weight by the beaming (Valid for super-Eddington RLO-BH systems in a GRRMHD framework).
- `Lx`: The X-ray luminosity of the binary. 
- `formation_channel`: Formation channel of the binary. Gives information about the evolutional steps of the system.
</details>

<hr style="border: none; height: 3px;
background: linear-gradient(to right, #0f23fa, #0fe7fa);">

## 4) Exploring the synthetic XRB population


In this section are presented the different steps to create cumulative X-ray luminosity function of the transient population.

### 4.1 Weight calculation 


To create an XLF for a given popualtion, we need first to normalize our population in order to have the cumulative XLF "N (> $L_{X}$) / ($M_{\odot} yr^{-1}$)". 

The function `Stat_XLF()`calculate the weight that represent each binary and is used this way:


<div align="center">

```python

Weigths = Stat_XLF(Transient_Pop, stat_kwargs=None)

```

</div>

If no statistical kwargs are provided, the default value stored `saved_ini_parameters` in the `posydon.popsyn.binarypopulation` file are used. If some kwargs are provided, they must be into a dictionnary form (see list below for which parameters can be passed). They are used as follow:

<div align="center">

```python

stat_kwargs = {'primary_mass_min': 0.01, 'primary_mass_max' : 200.0, 'q_min': 0.05, 'q_max' : 1}

Weights = Stat_XLF(Transient_Pop, stat_kwargs=stat_kwargs)

```
</div>

<details>
<summary> Population parameters list </summary>

- `number_of_binaries`
- `binary_fraction_scheme` and `binary_fraction_const`
- `star_formation`
- `max_simulation_time`
- `primary_mass_scheme`, `primary_mass_min` and `primary_mass_max`
- `secondary_mass_scheme`, `secondary_mass_min` and `secondary_mass_max`
- `orbital_scheme`, `orbital_period_scheme`, `orbital_period_min` and `orbital_period_max`
- `eccentricity_scheme`
- `q_min` and `q_max`
</details>

### 4.2 Visualizing the synthetic population

Once we have the weight of each binary, we can use a plotting function. The main function is `Plot_cumulative_XLF()` and does everything from the style of the plot (line sytle, line width, ...) to the smallest details. Below are shown two methods on how to use this function: once for the visualization of a single population and the other one when comparing two population.

- ***<u>Visualization of a single population XLF</u>***

We provide the transient population that we have created:

<div align="center">

```python

Plot_cumulative_XLF(Transient_Pop, weight, SFR=1.5, title="Title of the figure", ax=None, pad=60, figsize=None, show=True, **plot_kwargs)

```
</div>

This will create a figure containing the "Title of the figure" at position `pad=60`.


<details>
<summary> Keyword arguments </summary>

<div style="
padding:10px;
border-left:4px solid #06b6d4;
background-color:rgba(6,182,212,0.08);
border-radius:6px;
">

<b>Optional plotting arguments</b>

- <code>norm_factor</code>: normalization factor for Chandra-band luminosities  
- <code>extend_to_edges</code>: extends the CCDF to plot boundaries

</div>
</details>

- ***<u>Comparing multiple population XLFs</u>***

Here is an example on how to use `Plot_cumulative_XLF()` for comparing two figures:

<details>
<summary> Example of figures comparison </summary>

```python
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6.3, 6.3 * 0.618), sharey=True)


Plot_cumulative_XLF(Trans_Pop1_, Weights1,
    SFR = 1.2,
    title='Population 1',
    ax=ax1,
    show=False,
    linewidth=2.0
    )

Plot_cumulative_XLF(Trans_Pop2, Weights2,
    SFR = 1.2,
    title='Population 2',
    ax=ax2,
    show=False,
    linewidth=2.0
    )


# --- Post-processing additions ---
for ax in (ax1, ax2):

    ax.set_xlabel(r'$\log_{10}\left(L_{\mathrm{X}}\,/\,\mathrm{erg\,s^{-1}}\right)$', fontsize=9)
    ax.set_xticks(np.arange(35, 43, 1))
    ax.set_ylabel(r'$N(>L_{\mathrm{X}})\,/\,(\mathrm{M}_\odot\,\mathrm{yr}^{-1})$', fontsize=9)
    
    ax.tick_params(axis='both', which='both', labelsize=8, top=False, right=False)
    ax.set_title('')

    plot_Lehmer19_XLF(ax, 'path_to_data_file/J_ApJS_243_3_table7.dat.fits', SFR_total=45.4, color='#FFB2B2', label='Lehmer+19', linestyle='-',
                            linewidth=2.0)
    
    ax.legend(loc='lower center', 
              bbox_to_anchor=(0.5, 1.02),
              ncol=2, 
              fontsize=7, 
              handlelength=1.2, 
              borderpad=0.3,
              frameon=True,
              edgecolor='black',
              facecolor='white',
              fancybox=True)


    ax.get_legend().get_frame().set_linewidth(0.8)

ax2.set_ylabel('')

ax1.set_title('Transient population 1', fontsize=9, pad=55)
ax2.set_title('Transient population 2', fontsize=9, pad=55)

fig.tight_layout()
plt.show()
```

</details>

In this above example, we have used two axis, `ax1` and `ax2`. It can be extended with even more axis.

- ***<u>Comparison with observations (`Lehmer+19`)</u>***

We can overplot the observed XLF from `Lehmer+19` using its own plotting function `plot_Lehmer19_XLF()`. In this case, we cannot use the method used in *Visualization of a single population XLF* but we need to do it as follow:

<details>
<summary> Single plot XLF with Observed XLF </summary>

```python
fig, ax = Plot_cumulative_XLF(Transient_pop, weight, title = 'XLF of the default XRB population', show=False, pad=60, linewidth=2.0)

plot_Lehmer19_XLF(ax, 'path_to_data_file/J_ApJS_243_3_table7.dat.fits', SFR_total=45.4, color='#FFB2B2', label='Lehmer+19', linestyle='-', linewidth=2.5)

ax.legend(loc='lower center', 
          bbox_to_anchor=(0.5, 1.02), 
          bbox_transform=ax.transAxes, 
          ncol=3, 
          frameon=True, 
          fancybox=True, 
          edgecolor='k', 
          handlelength=1.5, 
          borderpad=0.4, 
          fontsize=9,
          )

ax.set_title(label='XLF of the XRB population', fontsize=11, pad=67)

plt.show()

```

</details>

### 4.3 Visualizing a sub-population from a global population

As the above function is used to plot a full population, we would like to know how to plot a sub-population. For that, we use the function `plot_ccdf()` which is the main function allowing `Plot_cumulative_XLF()` to plot the whole population. Here we compute the function as follow: 


<div align="center">

```python 
plot_ccdf(Transient_Pop, ax, weight, mask, label, SFR=1.5, **ccdf_kwargs)
```
</div>

To use this function, we must provide an axis on which to plot as well as a mask to filter out a sub-population from the main binary population.
Below is an example on how to use this function that shows the sub-population of black holes accreting through Roche lobe ooverflow from a hydrogen rich companion star (RLO-BH H-rich).

<details>
<summary> Code to plot a single sub-population </summary>

<div align="left">

```python

X_ray_range_Chandra_band = (np.log10(Transient_Pop.population['Lx'] * 0.5) >= 35) & (np.log10(Transient_Pop.population['Lx'] * 0.5) <= 43)

is_RLO = (Transient_Pop.population['System_type'] == 'RLO')
is_BH = (Transient_Pop.population['Accretor_state'] == 'BH')
is_H1rich = (Transient_Pop.population['Donor_state'].str.contains('H-rich|accreted_He', regex=True, na=False))

mask = is_RLO & is_BH & is_H1rich

fig, ax = plt.subplots(1,1, figsize=(6,4))

plot_ccdf(Transient_Pop, ax, weights, X_ray_range_Chandra_band & mask, 'RLO-BH H-rich', SFR = 1.5, linestyle='-', color='red')

ax.set_title('XLF of RLO-BH with H-rich companion')

ax.set_xlabel(r'$\log_{10}\left(L_{\mathrm{X},\,0.5-8\,\mathrm{keV}} \,/\, \mathrm{erg\,s^{-1}}\right)$', fontsize=11)
ax.set_xlim(35, 43)
ax.set_ylabel(r'$N(> L_{\mathrm{X},\,0.5-8\,\mathrm{keV}})\,/\,(\mathrm{M}_\odot\,\mathrm{yr}^{-1})$', fontsize=11)
ax.set_yscale('log')
ax.set_ylim(35, 43)
ax.legend()

plt.show()
```

</div>
</details>

 <div style="
padding: 12px;
border-left: 4px solid #0ea5e9;
background-color: rgba(14,165,233,0.08);
border-radius: 6px;
">

<b>Physical note:</b>  

 Following **Misra et al. (A&A 672, A99, 2023)**, we need to multiply our population by `0.5` in order to have the correct Chandra band.
 For more details about the function personalisation, see the definiton of the function.

</div>
 